In [5]:
import os
from typing import List, Tuple, Dict, Set, Optional, Union
from PIL import Image
import itertools
from collections import defaultdict
import numpy as np
import random

class CliqueForestDesertClassifier:
    def __init__(self, similarity_threshold: float = 0.25) -> None:
        """Инициализация классификатора"""
        self.similarity_threshold: float = similarity_threshold
        self.forest_cliques: List[np.ndarray] = []
        self.desert_cliques: List[np.ndarray] = []

    def build_similarity_graph(self, features: List[np.ndarray]) -> Dict[int, Set[int]]:
        """Строим граф схожести на основе нормализованного расстояния"""
        n: int = len(features)
        graph: Dict[int, Set[int]] = {i: set() for i in range(n)}
        
        for i, j in itertools.combinations(range(n), 2):
            # Нормализованное евклидово расстояние
            distance: float = np.sqrt(sum(
                ((a - b) / (np.std([features[i][k], features[j][k]]) + 1e-6)) ** 2 
                for k, (a, b) in enumerate(zip(features[i], features[j]))
            ))
            
            if distance < self.similarity_threshold:
                graph[i].add(j)
                graph[j].add(i)
        return graph

    def bron_kerbosch(self, graph: Dict[int, Set[int]]) -> List[Set[int]]:
        """Реализация алгоритма Брона-Кербоша с поиском максимальных клик"""
        all_cliques: List[Set[int]] = []
        
        def bk(R: Set[int], P: Set[int], X: Set[int]) -> None:
            if not P and not X:
                if len(R) >= 2:
                    all_cliques.append(R)
                return
            
            for v in list(P):
                neighbors: Set[int] = graph[v]
                bk(R | {v}, P & neighbors, X & neighbors)
                P.remove(v)
                X.add(v)
        
        bk(set(), set(graph.keys()), set())
        return all_cliques

    def train(self, image_paths: List[str], labels: List[str]) -> None:
        """Обучение модели"""
        features: List[np.ndarray] = []
        valid_labels: List[str] = []
        
        # Извлекаем признаки только для валидных изображений
        for path, label in zip(image_paths, labels):
            feat: Optional[np.ndarray] = extract_features(path)
            if feat is not None:
                features.append(feat)
                valid_labels.append(label)
        
        # Строим граф схожести
        graph: Dict[int, Set[int]] = self.build_similarity_graph(features)
        
        # Находим клики
        cliques: List[Set[int]] = self.bron_kerbosch(graph)
        
        # Анализируем каждую клику
        for clique in cliques:
            # Определяем доминирующий класс в клике
            class_counts: Dict[str, int] = defaultdict(int)
            for i in clique:
                class_counts[valid_labels[i]] += 1
            dominant_class: str = max(class_counts, key=class_counts.get)
            
            # Вычисляем средние признаки клики
            clique_feature: np.ndarray = np.mean([features[i] for i in clique], axis=0)
            
            # Сохраняем
            if dominant_class == "forest":
                self.forest_cliques.append(clique_feature)
            else:
                self.desert_cliques.append(clique_feature)

    def predict(self, image_path: str) -> Optional[str]:
        """Предсказание класса для нового изображения"""
        features: Optional[np.ndarray] = extract_features(image_path)
        if features is None:
            return None
        
        # Вычисляем расстояние до ближайшей клики каждого класса
        def get_min_distance(feature: np.ndarray, cliques: List[np.ndarray]) -> float:
            if cliques:
                return min(
                    np.sqrt(sum((a - b)**2 for a, b in zip(feature, clique)))
                    for clique in cliques
                )
            else: 
                return float("inf")
        
        forest_dist: float = get_min_distance(features, self.forest_cliques)
        desert_dist: float = get_min_distance(features, self.desert_cliques)
        
        if forest_dist < desert_dist:
            return "forest"
        else:
            return "desert"

    def evaluate(self, X_test: List[str], y_test: List[str]) -> float:
        """Оценка точности на тестовых данных"""
        correct: int = 0
        for path, true_label in zip(X_test, y_test):
            pred: Optional[str] = self.predict(path)
            if pred == true_label:
                correct += 1
                
        if X_test:
            return correct / len(X_test)
        else:
            return 0

def extract_features(image_path: str) -> Optional[np.ndarray]:
    """Извлечение признаков изображения"""
    try:
        img: Image.Image = Image.open(image_path).convert("RGB")
        img_array: np.ndarray = np.array(img)
        
        brightness: float = np.mean(img_array) / 255.0
        green_ratio: float = np.mean(img_array[:, :, 1]) / 255.0
        gray_img: Image.Image = img.convert("L")
        contrast: float = np.std(np.array(gray_img)) / 255.0
        
        return np.array([brightness, green_ratio, contrast])
    except Exception as e:
        print(f"Ошибка обработки {image_path}: {str(e)}")
        return None

def train_test_split(
    X: List[str], 
    y: List[str], 
    test_size: float = 0.2, 
    random_seed: Optional[int] = None
) -> Tuple[List[str], List[str], List[str], List[str]]:
    """Разделения данных на train/test"""
    if random_seed is not None:
        random.seed(random_seed)
        
    # Объединяем данные и перемешиваем
    combined: List[Tuple[str, str]] = list(zip(X, y))
    random.shuffle(combined)
    
    # Разделяем
    split_idx: int = int(len(combined) * (1 - test_size))
    train: List[Tuple[str, str]] = combined[:split_idx]
    test: List[Tuple[str, str]] = combined[split_idx:]
    
    # Разъединяем обратно
    X_train: List[str] = [x for x, y in train]
    y_train: List[str] = [y for x, y in train]
    X_test: List[str] = [x for x, y in test]
    y_test: List[str] = [y for x, y in test]
    
    return X_train, X_test, y_train, y_test

def load_dataset(dataset_path: str) -> Tuple[List[str], List[str]]:
    """Загрузка данных из папок forest и desert"""
    X: List[str] = []
    y: List[str] = []
    for class_name in ["forest", "desert"]:
        class_dir: str = os.path.join(dataset_path, class_name)
        files: List[str] = [f for f in os.listdir(class_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
            
        X.extend([os.path.join(class_dir, f) for f in files])
        y.extend([class_name] * len(files))
    return X, y

In [8]:
# Конфигурация
data_dir = "origins/data"
test_size = 0.2
random_seed = 52

# Загрузка данных
X, y = load_dataset(data_dir)

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_seed=52
)

# Создание и обучение классификатора
classifier = CliqueForestDesertClassifier(similarity_threshold=0.3)
classifier.train(X_train, y_train)

# Оценка точности
train_accuracy = classifier.evaluate(X_train, y_train)
test_accuracy = classifier.evaluate(X_test, y_test)
print(f"\nТочность на обучающей выборке: {train_accuracy*100:.2f}%")
print(f"Точность на тестовой выборке: {test_accuracy*100:.2f}%")


Точность на обучающей выборке: 79.10%
Точность на тестовой выборке: 76.40%


In [7]:
# Пример предсказания
if X_test:
    test_image = X_test[1]
    print(f"\nПример тестового изображения: {test_image}")
    print(f"Предсказанный класс: {classifier.predict(test_image)}")
    print(f"Истинный класс: {y_test[1]}")


Пример тестового изображения: origins/data\forest\forest.142.jpg
Предсказанный класс: forest
Истинный класс: forest
